In [1]:
import torch
import accelerate
print(torch.__version__, accelerate.__version__)

2.6.0+cpu 1.7.0


In [2]:
import sentencepiece
print(sentencepiece.__version__)


0.2.0


In [3]:
from transformers import T5Tokenizer, T5ForConditionalGeneration, pipeline

question_generation_model = "valhalla/t5-small-qg-prepend"
tokenizer = T5Tokenizer.from_pretrained(question_generation_model)
model = T5ForConditionalGeneration.from_pretrained(question_generation_model)

qg_pipeline = pipeline(
    "text2text-generation",
    model=model,
    tokenizer=tokenizer,
    num_beams=3
)

print("成功！QG pipeline 已建立。")


You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565
Device set to use cpu


成功！QG pipeline 已建立。


In [4]:
#!pip install transformers pypdf sentence-transformers datasets
#!pip install gradio

In [5]:
import pyarrow
print(pyarrow.__file__)
print(pyarrow.__version__)


C:\Users\wayne\anaconda3\envs\hfqa\lib\site-packages\pyarrow\__init__.py
20.0.0


In [6]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, pipeline
from pypdf import PdfReader
import re
from datasets import Dataset


# Question Generation

In [8]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, pipeline

question_generation_model = "valhalla/t5-small-qg-prepend"
tokenizer = AutoTokenizer.from_pretrained(question_generation_model, use_fast=False)
model = AutoModelForSeq2SeqLM.from_pretrained(question_generation_model)

qg_pipeline = pipeline(
    "text2text-generation",
    model=model,
    tokenizer=tokenizer,
    num_beams=3
)


Device set to use cpu


## Data Extraction

In [10]:
# Function to extract text from a PDF
def extract_text_from_pdf(pdf_path):
    reader = PdfReader(pdf_path)
    text = ""
    for page in reader.pages:
        text += page.extract_text()
    return text

# Function to clean and preprocess text
def clean_text(text):
    text = re.sub(r"\s+", " ", text)  # Remove extra whitespace
    text = re.sub(r"[^a-zA-Z0-9.,!?;'\s]", "", text)  # Remove special characters
    return text.strip()

# Function to split text into meaningful chunks
def split_text_into_chunks(text, max_tokens=200):
    sentences = re.split(r'(?<=[.!?]) +', text)  # Split by sentence boundaries
    chunks = []
    current_chunk = ""
    for sentence in sentences:
        if len(current_chunk.split()) + len(sentence.split()) <= max_tokens:
            current_chunk += " " + sentence
        else:
            chunks.append(current_chunk.strip())
            current_chunk = sentence
    if current_chunk:
        chunks.append(current_chunk.strip())
    return chunks

# Extract QA pairs from chunks and create a dataset
from datasets import Dataset

def extract_qa_pairs_and_create_dataset(pdf_path):
    raw_text = extract_text_from_pdf(pdf_path)
    cleaned_text = clean_text(raw_text)
    chunks = split_text_into_chunks(cleaned_text)

    # Generate QA pairs in batches
    input_texts = [f"generate questions: {chunk}" for chunk in chunks]
    generated = qg_pipeline(input_texts, max_length=64, num_return_sequences=3)

    # Prepare data for the HuggingFace dataset
    data = []
    for i, batch in enumerate(generated):
        for output in batch:  # Iterate through each output in the batch
            question = output['generated_text']
            context = chunks[i]  # Get the corresponding chunk
            data.append({"question": question, "context": context})

    return Dataset.from_list(data)

pdf_path = "example.pdf"  # Replace with your PDF file path
qa_dataset = extract_qa_pairs_and_create_dataset(pdf_path)
print(qa_dataset)


Both `max_new_tokens` (=256) and `max_length`(=64) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=64) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=64) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=64) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both

Dataset({
    features: ['question', 'context'],
    num_rows: 339
})


In [11]:
from pprint import pprint
pprint(qa_dataset[:5])


{'context': ['naturalresources.canada.ca '
             'ournaturalresourcesenergysourcesdistributionelectricityinfrastru '
             'Powering Canadas Future A Clean Electricity Strategy 155197 '
             'minutes Table of Contents Foreword  Clean Electricity Strategy '
             '1.0 The Case for Clean Electricity 1.1 Laying Out a Clean '
             'Electricity Strategy for Canada 1.2 A Strategy informed by '
             'extensive engagement, electricity sector experts, and Indigenous '
             'energy leaders 1.3 Key Guiding Principles 2.0 Toward the Grid of '
             'the Future 2.1 Global Context 2.2 Canadian Context 2.3 Regional '
             'Context 3.0 Federal Action 3.1 Focus Area 1 Growing the Grid and '
             'Managing Demand 3.2 Focus Area 2 Providing Policy Certainty and '
             'Smoothing the Path 3.3 Focus Area 3 Collaborating on Tailored '
             'Approaches for Every Region 4.0 Next Steps Annex 1  Canada '
             'El

## Embeddings

In [13]:
from sentence_transformers import SentenceTransformer

# Load the embedding model
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

# Generate embeddings for the dataset
def generate_embeddings(dataset):
    # Encode questions and contexts in batches
    question_embeddings = embedding_model.encode(dataset["question"], convert_to_tensor=True, batch_size=16)
    context_embeddings = embedding_model.encode(dataset["context"], convert_to_tensor=True, batch_size=16)
    return question_embeddings, context_embeddings

# Example: Generate embeddings
question_embeddings, context_embeddings = generate_embeddings(qa_dataset)
print("Embeddings generated successfully!")

Embeddings generated successfully!


In [14]:
print(question_embeddings.shape)   # (339, 384)  # 如果有339筆，每筆384維
print(context_embeddings.shape)    # (339, 384)

print(question_embeddings[0])      # 看第一筆的embedding


torch.Size([339, 384])
torch.Size([339, 384])
tensor([-4.9881e-02,  7.6088e-02,  4.9646e-02,  2.2475e-02,  1.1842e-02,
        -1.7303e-02, -4.5145e-02, -5.5412e-02, -3.1211e-02,  2.1702e-02,
        -1.3297e-02,  3.9568e-03,  1.5966e-03, -3.9743e-02, -5.1287e-02,
         2.8449e-02, -1.5933e-02, -2.2879e-02,  4.3017e-02, -2.1778e-02,
         2.6050e-02, -2.0943e-02,  7.1851e-02, -5.3243e-02,  7.8471e-02,
         6.4194e-02, -2.8760e-02,  1.1887e-02, -2.7563e-02,  1.4019e-02,
        -8.4122e-03, -1.8914e-02, -1.3566e-02,  3.6094e-03,  9.2956e-03,
         1.1128e-01, -5.2997e-02, -2.5509e-03,  6.5622e-02,  3.9739e-02,
        -2.7103e-02, -6.8642e-02, -2.9819e-02,  3.9888e-02, -1.0450e-01,
         9.9898e-03,  1.6665e-03, -4.2235e-02,  1.0909e-03, -6.2271e-02,
         5.0211e-02,  2.1722e-02, -3.9816e-02, -6.4433e-02,  9.8482e-02,
        -7.1374e-03, -1.1156e-02, -8.4512e-02,  1.3039e-01,  6.0762e-02,
         7.0821e-03, -8.1213e-02, -8.0293e-02, -1.8068e-02,  4.0624e-02,
     

## QA formatting

In [16]:
# Format the dataset for GPT-2 fine-tuning
def format_qa_for_gpt2(dataset):
    formatted_data = []
    for question, context in zip(dataset["question"], dataset["context"]):
        formatted_data.append({
            "prompt": f"Question: {question}\nAnswer:",
            "completion": context
        })
    return Dataset.from_list(formatted_data)

# Example: Format dataset
formatted_qa_dataset = format_qa_for_gpt2(qa_dataset)
formatted_qa_dataset.save_to_disk("formatted_qa_dataset")
print("Formatted dataset saved to disk.")

Saving the dataset (0/1 shards):   0%|          | 0/339 [00:00<?, ? examples/s]

Formatted dataset saved to disk.


## Training

In [18]:
from transformers import GPT2LMHeadModel, GPT2Tokenizer, Trainer, TrainingArguments, DataCollatorForLanguageModeling
from datasets import load_from_disk
import wandb
wandb.init(project="gpt2-finetune", mode="disabled")

# Load formatted dataset
formatted_qa_dataset = load_from_disk("formatted_qa_dataset")

# Tokenizer and model setup
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token
model = GPT2LMHeadModel.from_pretrained("gpt2")

# Tokenize the dataset
def tokenize_function(examples):
    return tokenizer(
        examples["prompt"] + examples["completion"],
        truncation=True,
        padding="max_length",
        max_length=512
    )

tokenized_dataset = formatted_qa_dataset.map(tokenize_function, batched=True, remove_columns=formatted_qa_dataset.column_names)

# Define training arguments
training_args = TrainingArguments(
    output_dir="./fine_tuned_gpt2",
    overwrite_output_dir=True,
    num_train_epochs=3,
    per_device_train_batch_size=8,
    save_steps=500,
    save_total_limit=2,
    logging_dir="./logs",
    logging_steps=100,
    learning_rate=5e-5
)

# Define data collator
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False  # Causal language modeling
)

# Initialize the trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=data_collator
)

# Fine-tune the model
trainer.train()


wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.
`loss_type=None` was set in the config but it is unrecognised.Using the default loss: `ForCausalLMLoss`.


Step,Training Loss
100,3.357300
200,2.712300


TrainOutput(global_step=255, training_loss=2.923914232441023, metrics={'train_runtime': 3489.745, 'train_samples_per_second': 0.583, 'train_steps_per_second': 0.073, 'total_flos': 531467993088000.0, 'train_loss': 2.923914232441023, 'epoch': 3.0})

In [19]:
trainer.save_model("./fine_tuned_gpt2")
tokenizer.save_pretrained("./fine_tuned_gpt2")


('./fine_tuned_gpt2\\tokenizer_config.json',
 './fine_tuned_gpt2\\special_tokens_map.json',
 './fine_tuned_gpt2\\vocab.json',
 './fine_tuned_gpt2\\merges.txt',
 './fine_tuned_gpt2\\added_tokens.json')

In [20]:
# Test the fine-tuned GPT-2
fine_tuned_model = GPT2LMHeadModel.from_pretrained("./fine_tuned_gpt2")
fine_tuned_tokenizer = GPT2Tokenizer.from_pretrained("gpt2")

def ask_question_gpt2(question, model, tokenizer):
    input_text = f"Question: {question}\nAnswer:"
    input_ids = tokenizer.encode(input_text, return_tensors="pt")
    outputs = model.generate(input_ids, max_length=200, num_beams=3, early_stopping=True)
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

# Example question
question = "What is the document about?"
answer = ask_question_gpt2(question, fine_tuned_model, fine_tuned_tokenizer)
print("\nGenerated Answer:", answer)


The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.



Generated Answer: Question: What is the document about?
Answer: The Canada Electricity Advisory Council is a non-partisan, non-partisan, non-partisan, non-partisan, non-partisan, non-partisan, non-partisan, non-partisan, non-partisan, non-partisan, non-partisan, non-partisan, non-partisan, non-partisan, non-partisan, non-partisan, non-partisan, non-partisan, non-partisan, non-partisan, non-partisan, non-profit, non-profit, non-profit, non-profit, non-profit, non-profit, non-profit, non-profit, non-profit, non-profit, non-profit, non-profit, non-profit, non-profit, non-profit, non-profit, non-profit, non-profit, non-profit, non-profit, non-profit, non-profit, non-profit, non-profit, non-


## Hugging Face

In [22]:
#!pip install huggingface_hub


In [29]:
from huggingface_hub import login
login()


In [ ]:
from huggingface_hub import HfApi, HfFolder, Repository

# Set your repository name (change to a unique name for your model)
repo_name = "fine-tuned-gpt2-qa"  # Replace with your desired model name
hf_user_name=""
model_path = "./fine_tuned_gpt2"  # Path to your fine-tuned model directory

# Create a repository and upload the model
from huggingface_hub import create_repo, upload_folder

# Create a repository (private=True if you want it to be private)
create_repo(repo_name, private=False)

# Upload the model folder to the repo
upload_folder(
    folder_path=model_path,
    repo_id=f"{hf_user_name}/{repo_name}",  # Replace <your-username> with your Hugging Face username
    commit_message="Upload fine-tuned GPT-2 for QA tasks",
)

print(f"Model uploaded to: https://huggingface.co/{hf_user_name}/{repo_name}")


In [ ]:
from transformers import GPT2LMHeadModel, GPT2Tokenizer

# Load the fine-tuned model from the Hugging Face Hub
model = GPT2LMHeadModel.from_pretrained("mehrobo/fine-tuned-gpt2-qa")  # Replace <your-username>
tokenizer = GPT2Tokenizer.from_pretrained("mehrobo/fine-tuned-gpt2-qa")

# Test the model
def ask_question_gpt2(question, model, tokenizer):
    input_text = f"Question: {question}\nAnswer:"
    input_ids = tokenizer.encode(input_text, return_tensors="pt")
    outputs = model.generate(input_ids, max_length=200, num_beams=3, early_stopping=True)
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

# Example question
question = "What is the document about?"
answer = ask_question_gpt2(question, model, tokenizer)
print("\nGenerated Answer:", answer)


In [ ]:
question = "What is the document about?"
answer = ask_question_gpt2(question, model, tokenizer)
print("\nGenerated Answer:", answer)


## Gradio

In [ ]:
import gradio as gr
from transformers import GPT2LMHeadModel, GPT2Tokenizer
import torch

# Model initialization
 #Include these if running this cell on its own
# repo_name = ""
# hf_user_name = ""
model_name = f"{hf_user_name}/{repo_name}"

try:
    model = GPT2LMHeadModel.from_pretrained(model_name)
    tokenizer = GPT2Tokenizer.from_pretrained(model_name)

    # Add padding token if it doesn't exist
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
        model.config.pad_token_id = model.config.eos_token_id
except Exception as e:
    print(f"Error loading model: {e}")

def chat(input_text, history):
    try:
        # Input validation
        if not input_text.strip():
            return "Please enter a message.", history

        # Format chat history
        chat_context = ""
        if history:
            chat_context = "\n".join([f"User: {turn[0]}\nBot: {turn[1]}" for turn in history])
            chat_context += "\n"

        # Prepare input
        full_prompt = f"{chat_context}User: {input_text}\nBot:"

        # Generate response with better parameters
        input_ids = tokenizer.encode(full_prompt, return_tensors="pt", truncation=True, max_length=512)

        with torch.no_grad():
            outputs = model.generate(
                input_ids,
                max_length=200,
                num_beams=5,
                no_repeat_ngram_size=2,
                temperature=0.7,
                top_k=50,
                top_p=0.9,
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id,
                do_sample=True
            )

        # Decode and clean response
        bot_response = tokenizer.decode(outputs[0], skip_special_tokens=True)
        bot_response = bot_response.split("Bot:")[-1].strip()

        if not bot_response:
            bot_response = "I apologize, but I couldn't generate a meaningful response. Please try rephrasing your question."

        # Update history
        history.append((input_text, bot_response))

        # Format history for display
        formatted_history = "\n".join([f"User: {turn[0]}\nBot: {turn[1]}\n" for turn in history])

        return bot_response, formatted_history

    except Exception as e:
        print(f"Error in chat function: {e}")
        return f"An error occurred: {str(e)}", history

# Create Gradio interface
with gr.Blocks() as demo:
    gr.Markdown("# GPT-2 QA Chatbot")

    chatbot = gr.State([])

    with gr.Row():
        txt = gr.Textbox(
            label="Type your message",
            placeholder="Enter your question here...",
            lines=2
        )
        submit_btn = gr.Button("Send")

    with gr.Row():
        output = gr.Textbox(label="Bot Response", lines=4)

    with gr.Row():
        history_display = gr.Textbox(
            label="Chat History",
            lines=10,
            interactive=False
        )

    # Event handler
    submit_btn.click(
        fn=chat,
        inputs=[txt, chatbot],
        outputs=[output, history_display],
        api_name="chat"
    )

    # Clear textbox after sending
    submit_btn.click(lambda: "", None, txt)

# Launch the app
if __name__ == "__main__":
    demo.launch(debug=True)